# Quantizing and exporting the KWS model

This notebook loads the trained PyTorch model, converts it to a `.tflite` model with LiteRT Torch, verifies that the exported model produces similar outputs on a sample input, and optionally generates a C array for ESP-IDF / TFLite Micro.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from model import DSCNN

try:
    import litert_torch
except ImportError:
    print("Error: litert_torch is not installed. Run: pip install litert-torch", file=sys.stderr)
    raise


## Configuration

Set the checkpoint path and the input shape your model expects.

- For your original setup, use `N_MELS=64` and `TIME_FRAMES=101`.
- For the smaller embedded-friendly setup, change these to `40` and `49` if that is what the model was trained on.


In [ ]:
CHECKPOINT_PATH = Path("model_files/kws_model.pt")
EXPORT_DIR = Path("model_files")
OUTPUT_TFLITE = EXPORT_DIR / "kws_model.tflite"
OUTPUT_CC = EXPORT_DIR / "model_data.cc"
OUTPUT_H = EXPORT_DIR / "model_data.h"

NUM_CLASSES = 4
N_MELS = 64
TIME_FRAMES = 101

# Keep this as batch=1 for export / deployment.
SAMPLE_INPUT_SHAPE = (1, 1, N_MELS, TIME_FRAMES)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## Load the pre-trained model


In [ ]:
def load_checkpoint(model: nn.Module, checkpoint_path: Path) -> nn.Module:
    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        cleaned[k[7:] if k.startswith("module.") else k] = v

    model.load_state_dict(cleaned, strict=True)
    model.eval()
    return model


model = DSCNN(num_classes=NUM_CLASSES)
model = load_checkpoint(model, CHECKPOINT_PATH)

print("Model loaded successfully.")
print(model)


## Create a representative example input

The converter needs a sample input with the exact shape the model will see at inference time.


In [ ]:
sample_input = torch.randn(*SAMPLE_INPUT_SHAPE, dtype=torch.float32)

with torch.no_grad():
    torch_output = model(sample_input).detach().cpu().numpy()

print("Sample input shape:", tuple(sample_input.shape))
print("PyTorch output shape:", torch_output.shape)
print("PyTorch output:", torch_output)


## Convert the model to TFLite with LiteRT Torch


In [ ]:
edge_model = litert_torch.convert(model, (sample_input,))
edge_model.export(str(OUTPUT_TFLITE))

print(f"Saved TFLite model to: {OUTPUT_TFLITE}")


## Verify the exported model numerically


In [ ]:
edge_output = edge_model(sample_input)

print("LiteRT output shape:", np.array(edge_output).shape)
print("LiteRT output:", edge_output)

is_close = np.allclose(torch_output, edge_output, atol=1e-4, rtol=1e-4)
max_abs_diff = np.max(np.abs(torch_output - edge_output))

print("Outputs close:", is_close)
print("Max absolute difference:", max_abs_diff)


## Optional: generate a C array for ESP-IDF / TFLite Micro

This writes `model_data.cc` and `model_data.h` with an aligned `g_model` array.


In [ ]:
def write_tflite_as_c_array(
    input_file: Path,
    output_cc: Path,
    output_h: Path,
    array_name: str = "g_model",
    bytes_per_line: int = 12,
) -> None:
    data = input_file.read_bytes()

    with open(output_h, "w", encoding="utf-8") as f:
        f.write("#ifndef MODEL_DATA_H_\n")
        f.write("#define MODEL_DATA_H_\n\n")
        f.write("#include <cstdint>\n\n")
        f.write(f"extern const unsigned char {array_name}[];\n")
        f.write(f"extern const int {array_name}_len;\n\n")
        f.write("#endif  // MODEL_DATA_H_\n")

    with open(output_cc, "w", encoding="utf-8") as f:
        f.write('#include "model_data.h"\n\n')
        f.write(f"alignas(16) const unsigned char {array_name}[] = {{\n")

        for i, b in enumerate(data):
            if i % bytes_per_line == 0:
                f.write("    ")
            f.write(f"0x{b:02x}, ")
            if i % bytes_per_line == bytes_per_line - 1:
                f.write("\n")

        if len(data) % bytes_per_line != 0:
            f.write("\n")

        f.write("};\n\n")
        f.write(f"const int {array_name}_len = {len(data)};\n")


write_tflite_as_c_array(OUTPUT_TFLITE, OUTPUT_CC, OUTPUT_H)

print(f"Saved C array source to: {OUTPUT_CC}")
print(f"Saved C array header to: {OUTPUT_H}")


## Next steps

- Copy `model_data.cc` and `model_data.h` into your ESP-IDF component.
- Use `g_model` and `g_model_len` in your TFLite Micro test app.
- If the model is too large for ESP32-WROOM, reduce the feature size and/or the model size before exporting again.
